### there are 2 part which was written by AI. they are
1) all part reusing phase 2 logic
2) retained_households list in phase 3.2

### reusing phase 2 logic

In [1]:
import pandas as pd
import numpy as np
import kagglehub
import os

path = kagglehub.dataset_download("frtgnn/dunnhumby-the-complete-journey")

tx = pd.read_csv(os.path.join(path, "transaction_data.csv"))

tx.columns = tx.columns.str.lower()

tx.head()

,household_key,basket_id,day,product_id,quantity,sales_value,store_id,retail_disc,trans_time,week_no,coupon_disc,coupon_match_disc
0,2375,26984851472,1,1004906,1,1.39,364,-0.60,1631,1,0.0,0.0
1,2375,26984851472,1,1033142,1,0.82,364,0.00,1631,1,0.0,0.0
2,2375,26984851472,1,1036325,1,0.99,364,-0.30,1631,1,0.0,0.0
3,2375,26984851472,1,1082185,1,1.21,364,0.00,1631,1,0.0,0.0
4,2375,26984851472,1,8160430,1,1.50,364,-0.39,1631,1,0.0,0.0


In [2]:
def build_rfm(tx, cutoff_day):
    data = tx[tx["day"] <= cutoff_day]

    rfm = data.groupby("household_key").agg(
        last_day  = ("day", "max"),
        frequency = ("basket_id", "nunique"),
        monetary  = ("sales_value", "sum"),
    )
    rfm["recency"] = cutoff_day - rfm["last_day"]

    r_conditions = [rfm["recency"] <= 7, rfm["recency"] <= 14,
                    rfm["recency"] <= 30, rfm["recency"] <= 60]
    rfm["r_score"] = np.select(r_conditions, [5, 4, 3, 2], default=1)
    rfm["f_score"] = pd.qcut(rfm["frequency"], q=5, labels=[1, 2, 3, 4, 5]).astype(int)
    rfm["m_score"] = pd.qcut(rfm["monetary"], q=5, labels=[1, 2, 3, 4, 5]).astype(int)
    rfm["value_score"] = (rfm["f_score"] + rfm["m_score"]) / 2

    seg_conditions = [
        (rfm["r_score"] >= 4) & (rfm["value_score"] >= 4),
        (rfm["r_score"] >= 4) & (rfm["value_score"] >= 3),
        (rfm["r_score"] <= 3) & (rfm["value_score"] >= 3),
        (rfm["r_score"] <= 2) & (rfm["value_score"] < 3),
    ]
    seg_names = ["High-Value Active", "Mid-Value Active", "High-Value Lapsing", "Low-Value Lapsed"]
    rfm["segments"] = np.select(seg_conditions, seg_names, default="Low-Value Active")

    return rfm

In [3]:
build_rfm(tx, cutoff_day=711)["segments"].value_counts()

segments
High-Value Active     782
Low-Value Active      733
Mid-Value Active      418
Low-Value Lapsed      372
High-Value Lapsing    195
Name: count, dtype: int64

In [4]:
rfm_cutoff = build_rfm(tx, cutoff_day=669)

In [5]:
rfm_cutoff

,last_day,frequency,monetary,recency,r_score,f_score,m_score,value_score,segments
household_key,,,,,,,,,
1,660,79,3959.91,9,4,3,4,3.5,Mid-Value Active
2,668,45,1954.34,1,5,2,3,2.5,Low-Value Active
3,640,45,2594.30,29,3,2,4,3.0,High-Value Lapsing
4,627,30,1200.11,42,2,1,2,1.5,Low-Value Lapsed
5,589,38,749.09,80,1,2,2,2.0,Low-Value Lapsed
...,...,...,...,...,...,...,...,...,...
2496,662,60,4105.29,7,5,3,4,3.5,Mid-Value Active
2497,665,216,6874.67,4,5,5,5,5.0,High-Value Active
2498,666,159,2511.39,3,5,5,3,4.0,High-Value Active


### phase 3.1 – choose the churn window

In [6]:
tx

,household_key,basket_id,day,product_id,quantity,sales_value,store_id,retail_disc,trans_time,week_no,coupon_disc,coupon_match_disc
0,2375,26984851472,1,1004906,1,1.39,364,-0.60,1631,1,0.0,0.0
1,2375,26984851472,1,1033142,1,0.82,364,0.00,1631,1,0.0,0.0
2,2375,26984851472,1,1036325,1,0.99,364,-0.30,1631,1,0.0,0.0
3,2375,26984851472,1,1082185,1,1.21,364,0.00,1631,1,0.0,0.0
4,2375,26984851472,1,8160430,1,1.50,364,-0.39,1631,1,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...
2595727,1598,42305362535,711,92130,1,0.99,3228,0.00,1520,102,0.0,0.0
2595728,1598,42305362535,711,114102,1,8.89,3228,0.00,1520,102,0.0,0.0
2595729,1598,42305362535,711,133449,1,6.99,3228,0.00,1520,102,0.0,0.0
2595730,1598,42305362535,711,6923644,1,4.50,3228,-0.49,1520,102,0.0,0.0


In [7]:
tx_unduplicated = tx[["household_key", "day"]].drop_duplicates().sort_values(["household_key", "day"])

In [8]:
tx_unduplicated

,household_key,day
46996,1,51
71711,1,67
123815,1,88
140773,1,94
172266,1,101
...,...,...
2531288,2500,695
2543184,2500,698
2565279,2500,704
2574778,2500,706


In [9]:
gaps = tx_unduplicated.groupby("household_key")["day"].diff().dropna()

In [10]:
gaps

71711      16.0
123815     21.0
140773      6.0
172266      7.0
190170      7.0
           ... 
2531288     4.0
2543184     3.0
2565279     6.0
2574778     2.0
2581430     2.0
Name: day, Length: 223033, dtype: float64

In [11]:
print(gaps.describe())
print(gaps.quantile([0.90, 0.95, 0.98, 0.99]))

count    223033.000000
mean          6.943869
std          15.870607
min           1.000000
25%           2.000000
50%           3.000000
75%           7.000000
max         666.000000
Name: day, dtype: float64
0.90    14.0
0.95    22.0
0.98    40.0
0.99    61.0
Name: day, dtype: float64


In [12]:
(gaps > 42).mean()

np.float64(0.01817668237435716)

In [13]:
(gaps > 60).mean()

np.float64(0.010307891657288383)

Choosing the churn window (42 days)

To avoid an arbitrary threshold, I computed the gap in days between consecutive shopping days for every household (223,033 gaps).

Median gap: 3 days. 95th percentile: 22 days. 99th percentile: 61 days.
Only 1.8% of gaps exceed 42 days; 1.0% exceed 60 days.

A 22-day window (95th percentile) is too short: a typical household makes ~80 visits in two years, so a 5% tail means several normal breaks of 22+ days, and loyal customers on holiday would be labelled as churned. At 42 days, false churn labels drop to under 2% of normal gaps. Extending to 60 days only reduces this by 0.8 points but moves the cutoff 18 days earlier, so 42 days (6 weeks) is the better trade-off.

### phase 3.2 – select active households at the cutoff and create churn labels

In [14]:
rfm_active = rfm_cutoff[rfm_cutoff.recency < 42]
rfm_active

,last_day,frequency,monetary,recency,r_score,f_score,m_score,value_score,segments
household_key,,,,,,,,,
1,660,79,3959.91,9,4,3,4,3.5,Mid-Value Active
2,668,45,1954.34,1,5,2,3,2.5,Low-Value Active
3,640,45,2594.30,29,3,2,4,3.0,High-Value Lapsing
6,669,236,5702.51,0,5,5,5,5.0,High-Value Active
7,660,51,2865.89,9,4,2,4,3.0,Mid-Value Active
...,...,...,...,...,...,...,...,...,...
2496,662,60,4105.29,7,5,3,4,3.5,Mid-Value Active
2497,665,216,6874.67,4,5,5,5,5.0,High-Value Active
2498,666,159,2511.39,3,5,5,3,4.0,High-Value Active


In [15]:
# List of households arriving between 670–711
retained_households = tx[tx.day > 669].household_key.unique()
rfm_active["churn"] = np.where(rfm_active.index.isin(retained_households), 0, 1)

In [16]:
rfm_active.churn.mean()

np.float64(0.0807799442896936)

In [17]:
rfm_active

,last_day,frequency,monetary,recency,r_score,f_score,m_score,value_score,segments,churn
household_key,,,,,,,,,,
1,660,79,3959.91,9,4,3,4,3.5,Mid-Value Active,0
2,668,45,1954.34,1,5,2,3,2.5,Low-Value Active,1
3,640,45,2594.30,29,3,2,4,3.0,High-Value Lapsing,0
6,669,236,5702.51,0,5,5,5,5.0,High-Value Active,0
7,660,51,2865.89,9,4,2,4,3.0,Mid-Value Active,0
...,...,...,...,...,...,...,...,...,...,...
2496,662,60,4105.29,7,5,3,4,3.5,Mid-Value Active,0
2497,665,216,6874.67,4,5,5,5,5.0,High-Value Active,0
2498,666,159,2511.39,3,5,5,3,4.0,High-Value Active,0


174 of 2,154 active households (8.1%) made no purchase in days 670–711 and are labelled as churned. The classes are imbalanced, so accuracy alone would be misleading.

### phase 3.3 - feature engineering

#### 3.3.a - visit frequency and trend

In [18]:
last_window =tx_unduplicated[tx_unduplicated.day.between(628, 669)]

In [19]:
prev_window = tx_unduplicated[tx_unduplicated.day.between(586, 627)]

In [20]:
prev_window_fillna = prev_window.groupby("household_key").size().reindex(rfm_active. index).fillna(0) 
prev_window_fillna

household_key
1        7.0
2        5.0
3        0.0
6       14.0
7        5.0
        ... 
2496     4.0
2497    10.0
2498    11.0
2499     7.0
2500     9.0
Length: 2154, dtype: float64

In [21]:
(prev_window_fillna == 0).mean()

np.float64(0.07288765088207985)

In [22]:
prev_window_fillna.median()

np.float64(5.0)

In [23]:
last_window_fillna = last_window.groupby("household_key").size().reindex(rfm_active. index).fillna(0)
last_window_fillna

household_key
1        5
2        1
3        1
6       15
7        8
        ..
2496     2
2497    16
2498    12
2499     2
2500    11
Length: 2154, dtype: int64

In [24]:
(last_window_fillna == 0).mean()

np.float64(0.0)

In [25]:
last_window_fillna.median()

np.float64(5.0)

In [26]:
rfm_active["visit_trend"] = last_window_fillna - prev_window_fillna

In [27]:
rfm_active

,last_day,frequency,monetary,recency,r_score,f_score,m_score,value_score,segments,churn,visit_trend
household_key,,,,,,,,,,,
1,660,79,3959.91,9,4,3,4,3.5,Mid-Value Active,0,-2.0
2,668,45,1954.34,1,5,2,3,2.5,Low-Value Active,1,-4.0
3,640,45,2594.30,29,3,2,4,3.0,High-Value Lapsing,0,1.0
6,669,236,5702.51,0,5,5,5,5.0,High-Value Active,0,1.0
7,660,51,2865.89,9,4,2,4,3.0,Mid-Value Active,0,3.0
...,...,...,...,...,...,...,...,...,...,...,...
2496,662,60,4105.29,7,5,3,4,3.5,Mid-Value Active,0,-2.0
2497,665,216,6874.67,4,5,5,5,5.0,High-Value Active,0,6.0
2498,666,159,2511.39,3,5,5,3,4.0,High-Value Active,0,1.0


In [28]:
rfm_active.groupby("churn").visit_trend.mean()

churn
0    0.075758
1    0.143678
Name: visit_trend, dtype: float64

In [29]:
rfm_active["last_window_fillna"] = last_window_fillna

In [30]:
rfm_active["prev_window_fillna"] = prev_window_fillna

In [31]:
rfm_active.groupby("churn").last_window_fillna.mean()

churn
0    7.193434
1    2.183908
Name: last_window_fillna, dtype: float64

In [32]:
rfm_active.groupby("churn").prev_window_fillna.mean()

churn
0    7.117677
1    2.040230
Name: prev_window_fillna, dtype: float64

In [33]:
rfm_active.groupby("churn").last_window_fillna.describe()

,count,mean,std,min,25%,50%,75%,max
churn,,,,,,,,
0,1980.0,7.193434,5.770800,1.0,3.0,6.0,10.0,38.0
1,174.0,2.183908,1.683162,1.0,1.0,2.0,3.0,11.0


In [34]:
rfm_active.groupby("churn").prev_window_fillna.describe()

,count,mean,std,min,25%,50%,75%,max
churn,,,,,,,,
0,1980.0,7.117677,5.889996,0.0,3.0,6.0,10.0,39.0
1,174.0,2.040230,2.336835,0.0,0.0,1.0,3.0,12.0


Visit trend does not separate churners. I compared visits in the 42 days before the cutoff (628–669) with the previous 42 days (586–627). Both groups are flat: churners went from 2.04 to 2.18 visits, non-churners from 7.12 to 7.19. The difference between the groups is not in the trend but in the level — churners visit roughly 2 times per 42 days, non-churners 7. Households here do not gradually slow down before leaving; low-frequency households simply stop. I therefore keep the visit count in the last 42 days as a feature and drop the trend.

The active-household filter requires at least one visit in days 628–669, so the last window has a floor of 1 while the previous window can be 0. This inflates the trend for low-frequency households, which are exactly the ones that churn — the positive trend among churners is an artefact of the sample selection, not behaviour.

#### 3.3.b - spending and basket size

In [35]:
tx

,household_key,basket_id,day,product_id,quantity,sales_value,store_id,retail_disc,trans_time,week_no,coupon_disc,coupon_match_disc
0,2375,26984851472,1,1004906,1,1.39,364,-0.60,1631,1,0.0,0.0
1,2375,26984851472,1,1033142,1,0.82,364,0.00,1631,1,0.0,0.0
2,2375,26984851472,1,1036325,1,0.99,364,-0.30,1631,1,0.0,0.0
3,2375,26984851472,1,1082185,1,1.21,364,0.00,1631,1,0.0,0.0
4,2375,26984851472,1,8160430,1,1.50,364,-0.39,1631,1,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...
2595727,1598,42305362535,711,92130,1,0.99,3228,0.00,1520,102,0.0,0.0
2595728,1598,42305362535,711,114102,1,8.89,3228,0.00,1520,102,0.0,0.0
2595729,1598,42305362535,711,133449,1,6.99,3228,0.00,1520,102,0.0,0.0
2595730,1598,42305362535,711,6923644,1,4.50,3228,-0.49,1520,102,0.0,0.0


In [36]:
spend_last_42d = (tx[tx["day"].between(628, 669)].groupby("household_key")["sales_value"].sum())
rfm_active["spend_last_42d"] = spend_last_42d
spend_last_42d

household_key
1       258.00
2       130.89
3        39.32
6       330.65
7       425.27
         ...  
2496     94.39
2497    770.34
2498    129.82
2499    177.86
2500    505.54
Name: sales_value, Length: 2154, dtype: float64

In [37]:
basket_size = spend_last_42d / last_window_fillna
rfm_active["basket_size"] = basket_size
basket_size

household_key
1        51.600000
2       130.890000
3        39.320000
6        22.043333
7        53.158750
           ...    
2496     47.195000
2497     48.146250
2498     10.818333
2499     88.930000
2500     45.958182
Length: 2154, dtype: float64

In [38]:
rfm_active.groupby("churn")[["spend_last_42d", "basket_size"]].mean()

,spend_last_42d,basket_size
churn,,
0,274.148778,39.257335
1,79.632241,38.391133


In [39]:
rfm_active.groupby("churn")[["spend_last_42d", "basket_size"]].describe()

spend_last_42d                                                           \
               count        mean         std   min     25%      50%       75%   
churn                                                                           
0             1980.0  274.148778  284.146308  2.19  78.500  180.435  371.0925   
1              174.0   79.632241  100.450015  0.69  16.135   49.955  101.5350   

               basket_size                                             \
           max       count       mean        std       min        25%   
churn                                                                   
0      2717.29      1980.0  39.257335  29.695467  0.751667  18.737861   
1       752.99       174.0  38.391133  40.371134  0.690000  12.315625   

                                      
             50%        75%      max  
churn                                 
0      31.769747  51.823250  296.810  
1      23.172500  48.840833  254.045

Spending adds nothing beyond visit frequency. Churners spend $80 in the last 42 days vs $274 for non-churners, but this is the same 3x gap already visible in visit counts. Spend per visit is essentially identical between the groups ($38.39 vs $39.26, against standard deviations of 30–40). Churning households do not shrink their baskets — they simply come less often.

#### 3.3.c - category variety

In [40]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("frtgnn/dunnhumby-the-complete-journey")

print("Path to dataset files:", path)

Path to dataset files: C:\Users\Enes\.cache\kagglehub\datasets\frtgnn\dunnhumby-the-complete-journey\versions\1


In [41]:
product = pd.read_csv(os.path.join(path, "product.csv"))
product.columns = product.columns.str.lower()

In [42]:
product

,product_id,manufacturer,department,brand,commodity_desc,sub_commodity_desc,curr_size_of_product
0,25671,2,GROCERY,National,FRZN ICE,ICE - CRUSHED/CUBED,22 LB
1,26081,2,MISC. TRANS.,National,NO COMMODITY DESCRIPTION,NO SUBCOMMODITY DESCRIPTION,
2,26093,69,PASTRY,Private,BREAD,BREAD:ITALIAN/FRENCH,
3,26190,69,GROCERY,Private,FRUIT - SHELF STABLE,APPLE SAUCE,50 OZ
4,26355,69,GROCERY,Private,COOKIES/CONES,SPECIALTY COOKIES,14 OZ
...,...,...,...,...,...,...,...
92348,18293142,6384,DRUG GM,National,BOOKSTORE,PAPERBACK BOOKS,
92349,18293439,6393,DRUG GM,National,BOOKSTORE,CHILDRENS LOW END,
92350,18293696,6406,DRUG GM,National,BOOKSTORE,PAPERBACK BEST SELLER,
92351,18294080,6442,DRUG GM,National,BOOKSTORE,PAPERBACK BOOKS,


In [43]:
product.commodity_desc.value_counts()

commodity_desc
GREETING CARDS/WRAP/PARTY SPLY    2785
CANDY - PACKAGED                  2475
MAKEUP AND TREATMENT              2467
HAIR CARE PRODUCTS                1744
SOFT DRINKS                       1704
                                  ... 
BOUQUET (NON ROSE)                   1
EASTER LILY                          1
MISCELLANEOUS(CORP USE ONLY)         1
PKG.SEAFOOD MISC                     1
FROZEN PACKAGE MEAT                  1
Name: count, Length: 308, dtype: int64

In [44]:
tx_last_42d = tx[tx["day"].between(628, 669)].merge(product, on="product_id", how="left")
tx_last_42d.shape

(174278, 18)

In [45]:
rfm_active["cm_desc_count"] = tx_last_42d.groupby("household_key")["commodity_desc"].nunique().reindex(rfm_active.index).fillna(0)

In [46]:
rfm_active["cm_desc_per_visit"] = rfm_active.cm_desc_count / last_window_fillna

In [47]:
rfm_active.cm_desc_per_visit

household_key
1       11.400000
2       44.000000
3        8.000000
6        3.200000
7       10.625000
          ...    
2496    13.500000
2497     5.625000
2498     2.000000
2499    17.000000
2500     6.181818
Name: cm_desc_per_visit, Length: 2154, dtype: float64

In [48]:
rfm_active.groupby("churn")[["cm_desc_count", "cm_desc_per_visit"]].mean()

,cm_desc_count,cm_desc_per_visit
churn,,
0,37.519192,6.434493
1,14.798851,7.730765


In [49]:
rfm_active.groupby("churn")[["cm_desc_count", "cm_desc_per_visit"]].describe()

cm_desc_count                                                      \
              count       mean        std  min   25%   50%   75%    max   
churn                                                                     
0            1980.0  37.519192  26.101759  1.0  16.0  33.0  54.0  144.0   
1             174.0  14.798851  14.043689  1.0   5.0  10.0  21.0   76.0   

      cm_desc_per_visit                                                       \
                  count      mean       std   min    25%       50%       75%   
churn                                                                          
0                1980.0  6.434493  4.770355  0.25  3.375  5.333333  8.000000   
1                 174.0  7.730765  7.589139  1.00  3.000  5.625000  9.833333   

             
        max  
churn        
0      44.0  
1      44.0

Category variety separates churners, but only through frequency. Churners touch 14.8 distinct commodity categories in the last 42 days vs 37.5 for non-churners. Per visit, however, the picture reverses and flattens (7.73 vs 6.43, medians 5.63 vs 5.33) — an artefact of low-frequency households having fewer repeat purchases. I keep the raw category count and drop the per-visit ratio.

#### 3.3.d - 	customer tenure

In [50]:
rfm_active["first_day"] = tx[tx.day.between(0, 669)].groupby("household_key").day.min().reindex(rfm_active.index)
rfm_active.first_day

household_key
1        51
2       103
3       113
6       118
7        23
       ... 
2496    117
2497     78
2498    105
2499     70
2500     79
Name: first_day, Length: 2154, dtype: int64

In [51]:
rfm_active["consumer_time"] = 669 - rfm_active.first_day

In [52]:
rfm_active.consumer_time

household_key
1       618
2       566
3       556
6       551
7       646
       ... 
2496    552
2497    591
2498    564
2499    599
2500    590
Name: consumer_time, Length: 2154, dtype: int64

In [53]:
rfm_active.groupby("churn")["consumer_time"].mean()

churn
0    603.319697
1    600.862069
Name: consumer_time, dtype: float64

In [54]:
rfm_active.groupby("churn")["consumer_time"].describe()

,count,mean,std,min,25%,50%,75%,max
churn,,,,,,,,
0,1980.0,603.319697,36.006320,236.0,573.0,599.0,634.0,668.0
1,174.0,600.862069,53.131242,56.0,573.0,604.5,627.0,668.0


Customer tenure is constant in this panel. Days since first purchase are almost identical for both groups (603 vs 601 on average, medians 599 vs 604). The dunnhumby panel follows a fixed set of 2,500 households over the full two years, so there are effectively no "new" customers whose shorter history could explain churn. The churners' higher variance (std 53 vs 36, min 56 days) comes from a handful of late joiners — too few to be useful. Dropped.

#### 3.3.e - campaign exposure

In [55]:
campaign_table = pd.read_csv(os.path.join(path, "campaign_table.csv"))
campaign_table.columns = campaign_table.columns.str.lower()
campaign_desc = pd.read_csv(os.path.join(path, "campaign_desc.csv"))
campaign_desc.columns = campaign_desc.columns.str.lower()

In [56]:
campaign_desc.head()

,description,campaign,start_day,end_day
0,TypeB,24,659,719
1,TypeC,15,547,708
2,TypeB,25,659,691
3,TypeC,20,615,685
4,TypeB,23,646,684


In [57]:
campaign_table.head()

,description,household_key,campaign
0,TypeA,17,26
1,TypeA,27,26
2,TypeA,212,26
3,TypeA,208,26
4,TypeA,192,26


In [58]:
campain_merged = campaign_table.merge(campaign_desc, on="campaign")
campain_merged

,description_x,household_key,campaign,description_y,start_day,end_day
0,TypeA,17,26,TypeA,224,264
1,TypeA,27,26,TypeA,224,264
2,TypeA,212,26,TypeA,224,264
3,TypeA,208,26,TypeA,224,264
4,TypeA,192,26,TypeA,224,264
...,...,...,...,...,...,...
7203,TypeC,1803,15,TypeC,547,708
7204,TypeC,1082,15,TypeC,547,708
7205,TypeC,942,15,TypeC,547,708
7206,TypeC,855,15,TypeC,547,708


In [59]:
rfm_active["campaign_count"] = campain_merged[campain_merged.start_day.between(0, 669)].groupby("household_key")["campaign"].nunique().reindex(rfm_active.index).fillna(0)

In [60]:
rfm_active.groupby("churn").campaign_count.mean()

churn
0    3.424242
1    0.977011
Name: campaign_count, dtype: float64

In [61]:
rfm_active.groupby("churn").campaign_count.describe()

,count,mean,std,min,25%,50%,75%,max
churn,,,,,,,,
0,1980.0,3.424242,3.315454,0.0,0.0,3.0,6.0,17.0
1,174.0,0.977011,1.865271,0.0,0.0,0.0,1.0,8.0


Campaign exposure is strongly associated with retention — but this is not causal evidence. Non-churners were included in 3.4 campaigns on average before the cutoff; churners in 1.0, with a median of zero. Campaigns were not randomly assigned: the retailer plausibly targeted households that were already frequent and valuable, so exposure may be a marker of being a good customer rather than a cause of staying. Phase 4 separates these two explanations with a proper incrementality design.

#### 3.3.f - coupon redemption

In [62]:
coupon_redempt = pd.read_csv(os.path.join(path, "coupon_redempt.csv"))
coupon_redempt.columns = coupon_redempt.columns.str.lower()

In [63]:
coupon_redempt

,household_key,day,coupon_upc,campaign
0,1,421,10000085364,8
1,1,421,51700010076,8
2,1,427,54200000033,8
3,1,597,10000085476,18
4,1,597,54200029176,18
...,...,...,...,...
2313,2496,592,54900050076,18
2314,2496,610,55100000013,18
2315,2500,449,53663200076,8
2316,2500,449,54300031076,8


In [64]:
coupon_redempt[coupon_redempt.day.between(0, 669)]

,household_key,day,coupon_upc,campaign
0,1,421,10000085364,8
1,1,421,51700010076,8
2,1,427,54200000033,8
3,1,597,10000085476,18
4,1,597,54200029176,18
...,...,...,...,...
2313,2496,592,54900050076,18
2314,2496,610,55100000013,18
2315,2500,449,53663200076,8
2316,2500,449,54300031076,8


In [65]:
rfm_active["coupon_used"] = coupon_redempt[coupon_redempt.day.between(0, 669)].groupby("household_key").size().reindex(rfm_active.index).fillna(0)

In [66]:
rfm_active.coupon_used

household_key
1        5.0
2        0.0
3        0.0
6        0.0
7        0.0
        ... 
2496    11.0
2497     0.0
2498     0.0
2499     0.0
2500     3.0
Name: coupon_used, Length: 2154, dtype: float64

In [67]:
rfm_active.groupby("churn").coupon_used.mean()

churn
0    1.092929
1    0.362069
Name: coupon_used, dtype: float64

In [68]:
rfm_active.groupby("churn").coupon_used.describe()

,count,mean,std,min,25%,50%,75%,max
churn,,,,,,,,
0,1980.0,1.092929,3.370719,0.0,0.0,0.0,0.0,32.0
1,174.0,0.362069,2.399650,0.0,0.0,0.0,0.0,29.0


In [69]:
rfm_active["coupon_user"] = (rfm_active["coupon_used"] > 0).astype(int)

In [70]:
rfm_active.groupby("churn").coupon_user.mean()

churn
0    0.206061
1    0.080460
Name: coupon_user, dtype: float64

Coupon redemption follows the same pattern as campaign exposure. 20.6% of retained households redeemed at least one coupon before the cutoff, against 8.0% of churners. Redemption counts are heavily zero-inflated (median 0 in both groups, max 32), so I use the binary indicator rather than the raw count. As with campaigns, this is association, not causation: redeeming a coupon requires visiting the store, so frequent shoppers mechanically redeem more. Phase 4 addresses this directly.

Summary of feature engineering. Four attempts to find a signal independent of visit frequency all failed: visit trend, spend per visit, categories per visit, and customer tenure. Each either collapsed into the frequency gap or was flat across the two groups. What survives is level, not shape — visits in the last 42 days (7.2 vs 2.2) and distinct categories bought (37.5 vs 14.8). In this dataset, how often a household shops is what predicts churn; how it shops does not.

### 3.4 phase 3.4 – feature selection and train-test split

In [73]:
rfm_active[["frequency","last_window_fillna","spend_last_42d","recency","monetary","cm_desc_count","campaign_count","coupon_user"]].corr(method = "spearman")

,frequency,last_window_fillna,spend_last_42d,recency,monetary,cm_desc_count,campaign_count,coupon_user
frequency,1.000000,0.757202,0.590717,-0.478944,0.816186,0.569819,0.787802,0.371415
last_window_fillna,0.757202,1.000000,0.776425,-0.638149,0.636256,0.736286,0.548516,0.316838
spend_last_42d,0.590717,0.776425,1.000000,-0.512133,0.760821,0.943369,0.562452,0.374427
recency,-0.478944,-0.638149,-0.512133,1.000000,-0.410548,-0.485956,-0.355108,-0.205085
monetary,0.816186,0.636256,0.760821,-0.410548,1.000000,0.722955,0.835231,0.459215
cm_desc_count,0.569819,0.736286,0.943369,-0.485956,0.722955,1.000000,0.548799,0.372784
campaign_count,0.787802,0.548516,0.562452,-0.355108,0.835231,0.548799,1.000000,0.452795
coupon_user,0.371415,0.316838,0.374427,-0.205085,0.459215,0.372784,0.452795,1.000000


In [75]:
features = ["recency", "last_window_fillna", "cm_desc_count", "campaign_count", "coupon_user"]
X = rfm_active[features]
y = rfm_active["churn"]

In [77]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)
# this line is written by AI

In [82]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# this line is written by AI

Feature selection. Several candidate features are near-duplicates of each other: spend in the last 42 days correlates 0.94 with the number of distinct categories bought, and total monetary value correlates 0.84 with campaign exposure and 0.82 with frequency. Leaving them all in would make the logistic coefficients unstable and uninterpretable, which defeats the purpose of running a linear model at all. I keep five features — recency, visits in the last 42 days, distinct categories, campaign exposure and a binary coupon-redemption flag. Long-run frequency is dropped in favour of the 42-day visit count, which measures current rather than historical behaviour. The highest remaining pairwise correlation is 0.74.

The 2,154 households are split 75/25, stratified on churn so both sides carry the same 8% rate (44 churners in the test set). Features are standardised with a scaler fitted on the training set only, so no test-set information leaks into training.

### 3.5 - logistic regression
##### this part is written by AI because i do not feel myself comfortably those tools

In [84]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(class_weight="balanced", max_iter=1000)
model.fit(X_train_scaled, y_train)

,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to

In [85]:
coefs = pd.DataFrame({
    "feature": features,
    "coef": model.coef_[0],
    "odds_ratio": np.exp(model.coef_[0])
}).sort_values("coef")

coefs

,feature,coef,odds_ratio
1,last_window_fillna,-1.778074,0.168963
3,campaign_count,-0.907413,0.403567
2,cm_desc_count,-0.417116,0.658944
4,coupon_user,0.359878,1.433155
0,recency,0.371803,1.450348


In [86]:
from sklearn.metrics import roc_auc_score, confusion_matrix, classification_report

y_proba = model.predict_proba(X_test_scaled)[:, 1]
y_pred = model.predict(X_test_scaled)

print("AUC:", roc_auc_score(y_test, y_proba))
print()
print(confusion_matrix(y_test, y_pred))
print()
print(classification_report(y_test, y_pred))

AUC: 0.8398071625344353

[[377 118]
 [ 10  34]]

              precision    recall  f1-score   support

           0       0.97      0.76      0.85       495
           1       0.22      0.77      0.35        44

    accuracy                           0.76       539
   macro avg       0.60      0.77      0.60       539
weighted avg       0.91      0.76      0.81       539



AUC 0.84 on the held-out test set. With class_weight="balanced" the model catches 34 of the 44 actual churners (77% recall) at the cost of 118 false alarms (22% precision) — a deliberate trade-off, since losing a customer costs more than sending an unnecessary coupon. Accuracy (76%) is not a meaningful metric here: predicting "no churn" for everyone would score 92%.

Visit count dominates. A one-standard-deviation increase in visits over the last 42 days cuts the odds of churning by a factor of six (odds ratio 0.17), far ahead of campaign exposure (0.40) and category variety (0.66).

Coupon redemption flips sign once frequency is controlled for. In the raw comparison, coupon users churned less (8.0% vs 20.6%). In the model, holding visit frequency constant, the coefficient is positive (odds ratio 1.43). Redeeming a coupon requires visiting the store, so the raw association was picking up frequency, not any effect of the coupon — a textbook case of omitted variable bias, and the clearest argument for the causal design in Phase 4.

### 3.6 - XGBoost
##### this part is written by AI because i do not feel myself comfortably those tools

In [87]:
import xgboost
print(xgboost.__version__)

3.2.0


In [88]:
from xgboost import XGBClassifier

xgb = XGBClassifier(
    scale_pos_weight=11.4,
    max_depth=3,
    n_estimators=200,
    learning_rate=0.1,
    random_state=42,
    eval_metric="logloss"
)
xgb.fit(X_train_scaled, y_train)

,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",'logloss'
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [89]:
y_proba_xgb = xgb.predict_proba(X_test_scaled)[:, 1]
y_pred_xgb = xgb.predict(X_test_scaled)

print("XGBoost AUC:", roc_auc_score(y_test, y_proba_xgb))
print("Logistic AUC:", roc_auc_score(y_test, y_proba))
print()
print(confusion_matrix(y_test, y_pred_xgb))
print()
print(classification_report(y_test, y_pred_xgb))

XGBoost AUC: 0.8095730027548209
Logistic AUC: 0.8398071625344353

[[415  80]
 [ 19  25]]

              precision    recall  f1-score   support

           0       0.96      0.84      0.89       495
           1       0.24      0.57      0.34        44

    accuracy                           0.82       539
   macro avg       0.60      0.70      0.61       539
weighted avg       0.90      0.82      0.85       539



In [90]:
pd.DataFrame({
    "feature": features,
    "importance": xgb.feature_importances_
}).sort_values("importance", ascending=False)

,feature,importance
1,last_window_fillna,0.511953
0,recency,0.155630
3,campaign_count,0.133502
2,cm_desc_count,0.116105
4,coupon_user,0.082810


XGBoost does not beat logistic regression here (AUC 0.810 vs 0.840). With 1,615 training rows, only 130 of them churners, and five features, there is no complex interaction structure for boosting to exploit, and the extra flexibility mostly fits noise. Recall drops from 77% to 57%. Both models agree on what matters: visits in the last 42 days carry 51% of XGBoost's feature importance and the largest logistic coefficient by a wide margin. I keep logistic regression as the final model — equal or better performance, and coefficients that can be read as business statements.